# VisionX V3 — Fine-Tuning 14 Kelas (Google Colab GPU)
Notebook ini disiapkan untuk melatih model VisionX V3 (14 kelas) menggunakan GPU T4 gratis di Google Colab.

**Langkah persiapan sebelum klik Run All:**
1. Pastikan Runtime Colab menggunakan GPU: Menu `Runtime` > `Change runtime type` > Pilih `T4 GPU` > `Save`.
2. Upload 2 file dari laptop ke Google Drive (My Drive / root):
   - `processed_v2.zip` (dataset siap pakai)
   - `visionx_v2_best.pt` (checkpoint awal fine-tune)

### Cell 1: Mount Google Drive & Install Ultralytics

In [ ]:
from google.colab import drive
import os, sys

# 1. Hubungkan Google Drive
drive.mount('/content/drive')

# 2. Install Ultralytics YOLO
!pip install -q ultralytics

import torch
print('\n--- Environment Check ---')
print('PyTorch Version:', torch.__version__)
print('CUDA Available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device     :', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'VRAM           : {vram:.2f} GB')
else:
    print('PERINGATAN: GPU belum aktif! Silakan buka Runtime > Change runtime type > Pilih T4 GPU.')


### Cell 2: Ekstrak Dataset processed_v2 dari Google Drive

In [ ]:
import zipfile, os, yaml

# Cari processed_v2.zip di Drive
ZIP_PATH = '/content/drive/MyDrive/processed_v2.zip'
if not os.path.exists(ZIP_PATH):
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'processed_v2.zip' in files:
            ZIP_PATH = os.path.join(root, 'processed_v2.zip')
            break

print(f'Menggunakan file: {ZIP_PATH}')
DEST_DIR = '/content/datasets/processed_v2'
os.makedirs(DEST_DIR, exist_ok=True)

print('Mengekstrak dataset ke /content/datasets/processed_v2...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(DEST_DIR)

print('Ekstraksi selesai!')

# Sesuaikan path di dataset.yaml ke direktori lokal Colab
yaml_path = os.path.join(DEST_DIR, 'dataset.yaml')
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['path'] = DEST_DIR
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print('\n--- Isi dataset.yaml terverifikasi ---')
with open(yaml_path) as f:
    print(f.read())


### Cell 3: Load Checkpoint Base & Verifikasi Transfer Bobot nc=14

In [ ]:
from ultralytics import YOLO

# Cari checkpoint visionx_v2_best.pt / best.pt di Drive
CHECKPOINT_PATH = '/content/drive/MyDrive/visionx_v2_best.pt'
if not os.path.exists(CHECKPOINT_PATH):
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        for f in files:
            if f in ['visionx_v2_best.pt', 'best.pt']:
                CHECKPOINT_PATH = os.path.join(root, f)
                break

print(f'Loading Base Checkpoint: {CHECKPOINT_PATH}')
model = YOLO(CHECKPOINT_PATH)
print('Kelas awal checkpoint (nc=7):', model.names)


### Cell 4: Training VisionX V3 (20 Epochs di GPU Colab)

In [ ]:
DATA_YAML = '/content/datasets/processed_v2/dataset.yaml'

print('Memulai fine-tuning VisionX V3 di GPU Colab...')
results = model.train(
    data=DATA_YAML,
    epochs=20,
    patience=6,
    batch=16,
    imgsz=640,
    optimizer='auto',
    lr0=0.005,
    lrf=0.01,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    project='runs/detect',
    name='visionx_v3',
    exist_ok=True,
    verbose=True,
    seed=42
)
print('\nTraining VisionX V3 20-Epoch Selesai!')


### Cell 5: Zip Hasil Training & Simpan Otomatis ke Google Drive

In [ ]:
import shutil, os

OUTPUT_ZIP_DRIVE = '/content/drive/MyDrive/visionx_v3_results.zip'
RUN_DIR = '/content/runs/detect/visionx_v3'

print(f'Mengompresi hasil training dari: {RUN_DIR}...')
shutil.make_archive('/content/visionx_v3_results', 'zip', RUN_DIR)

shutil.copy2('/content/visionx_v3_results.zip', OUTPUT_ZIP_DRIVE)
size_mb = os.path.getsize(OUTPUT_ZIP_DRIVE) / (1024 * 1024)
print(f'\nSUKSES! Hasil training tersimpan di Google Drive:')
print(f' -> {OUTPUT_ZIP_DRIVE} ({size_mb:.2f} MB)')
print('\nSilakan download visionx_v3_results.zip dari Drive ke laptop, lalu ekstrak isinya ke models/visionx_v3/')
